In [40]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
import gc

In [41]:
ds_sla = xr.open_dataset('SLA_UPDATED.nc')

In [42]:
ds_gfs = xr.open_dataset('GFS_UPDATED_V1.nc')

In [43]:
ocean_mask = ~ds_sla.sla.isel(time=0).isnull()

In [44]:
lat_idxs,lon_idxs = np.where(ocean_mask)

In [45]:
sla_now = ds_sla.sla

ds_ml = xr.Dataset({
        "sla_now": sla_now,
        
        **{
        f"sla_lag{i}": ds_sla.sla.shift(time=i)
        for i in range(1,15)
        },

        "tmp_surface": ds_gfs.TMP_surface,
        "tmp_2m": ds_gfs.TMP_2maboveground,

        "ugrd": ds_gfs.UGRD_10maboveground,
        "vgrd": ds_gfs.VGRD_10maboveground,

        "dswrf": ds_gfs.DSWRF_surface,
        "uswrf": ds_gfs.USWRF_surface,
        "dlwrf": ds_gfs.DLWRF_surface,
        "ulwrf": ds_gfs.ULWRF_surface,

        "spfh": ds_gfs.SPFH_2maboveground,
        "prate": ds_gfs.PRATE_surface,

        
    })
ds_ml["month"] = ds_ml.time.dt.month
ds_ml["dayofyear"] = ds_ml.time.dt.dayofyear

In [46]:
features = (
    ["sla_now"]
    + [f"sla_lag{i}" for i in range(1,15)]
    + [
        "tmp_surface",
        "tmp_2m",
        "ugrd",
        "vgrd",
        "dswrf",
        "uswrf",
        "dlwrf",
        "ulwrf",
        "spfh",
        "prate",
        "month",
        "dayofyear",
        "latitude",
        "longitude"
    ]
)

In [47]:
ds_ml = ds_ml.stack(
    point=("latitude","longitude")
)
df_features = (ds_ml.to_dataframe().reset_index()).dropna()

In [48]:
def build_Dataset(lead):

    target = (
        ds_sla.sla
        .shift(time=-lead)
        .where(ocean_mask)
        .stack(point=("latitude", "longitude"))
    )

    target_df = target.to_dataframe(name="target").reset_index()

    df = df_features.merge(
        target_df,
        on=["time", "latitude", "longitude"]
    )

    df = df.dropna(subset=["target"])

    # Chronological split
    train = df[df.time < "2023-01-01"]
    val   = df[(df.time >= "2023-01-01") & (df.time < "2024-01-01")]
    test  = df[df.time >= "2024-01-01"]

    X_train = train[features]
    y_train = train["target"]

    X_val = val[features]
    y_val = val["target"]

    X_test = test[features]
    y_test = test["target"]

    coords_test = test[["time", "latitude", "longitude"]]

    return (
        X_train, y_train,
        X_val, y_val,
        X_test, y_test,
        coords_test
    )

In [49]:
def train_and_evaluate(lead):

    (
        X_train, y_train,
        X_val, y_val,
        X_test, y_test,
        coords_test
    ) = build_Dataset(lead)

    model = XGBRegressor(
        n_estimators=250,   
        max_depth=8,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        device="cuda",
        n_jobs=-1,
        random_state=42
    )

    # Train
    model.fit(X_train, y_train)

    # Predict on validation set
    model.set_params(device="cpu")
    pred_val = model.predict(X_val)

    rmse_val = np.sqrt(
        mean_squared_error(y_val, pred_val)
    )

    # Predict on test set
    pred_xgb = model.predict(X_test)

    rmse_xgb = np.sqrt(
        mean_squared_error(y_test, pred_xgb)
    )

    pred_persistence = X_test["sla_now"]

    rmse_persistence = np.sqrt(
        mean_squared_error(y_test, pred_persistence)
    )

    pred_df = coords_test.copy()

    pred_df["truth"] = y_test.values
    pred_df["xgb"] = pred_xgb
    pred_df["pers"] = pred_persistence.values

    pred_df["err_xgb"] = pred_xgb - y_test.values
    pred_df["err_pers"] = pred_persistence.values - y_test.values

    rmse_map = (
        pred_df
        .groupby(["latitude", "longitude"])
        .agg(
            rmse_xgb=("err_xgb",
                      lambda x: np.sqrt(np.mean(x**2))),
            rmse_pers=("err_pers",
                       lambda x: np.sqrt(np.mean(x**2)))
        )
        .reset_index()
    )

    rmse_map["improvement"] = (
        rmse_map["rmse_pers"] -
        rmse_map["rmse_xgb"]
    )

    return (
        rmse_val,              # Validation RMSE
        rmse_xgb,              # Test RMSE
        rmse_persistence,      # Test persistence RMSE
        model,
        X_test,
        rmse_map,
        pred_df
    )

In [50]:
results = []

for lead in range(1, 8):

    (
        rmse_val,
        rmse_xgb,
        rmse_persistence,
        model,
        X_test,
        rmse_map,
        pred_df
    ) = train_and_evaluate(lead)

    results.append({
        "lead": lead,
        "validation": rmse_val,
        "xgb": rmse_xgb,
        "persistence": rmse_persistence
    })

    print(
        f"Lead {lead}: "
        f"Validation={rmse_val:.5f}, "
        f"Persistence={rmse_persistence:.5f}, "
        f"XGB={rmse_xgb:.5f}"
    )

    improvement_da = (
        rmse_map
        .set_index(["latitude", "longitude"])["improvement"]
        .to_xarray()
    )
    

    plt.figure(figsize=(12, 6))

    improvement_da.plot(
        cmap="RdBu_r",
        vmin=-0.01,
        vmax=0.01,
        cbar_kwargs={
            "ticks": np.linspace(-0.01, 0.01, 5)
        }
    )

    # Save RMSE table for this lead
    rmse_map.to_parquet(
        f"xgb_results/rmse_map_lead/rmse_map_lead_{lead}.parquet"
    )

    # Save trained XGBoost model
    model.save_model(
        f"xgb_results/model_info/xgb_lead_{lead}.json"
    )

    # Save gridded improvement map
    improvement_da.to_netcdf(
        f"xgb_results/improv_leadnc/improvement_lead_{lead}.nc"
    )

    # Save predictions
    pred_df.to_parquet(
        f"xgb_results/pred_lead/predictions_lead_{lead}.parquet"
    )

    plt.title(
        f"RMSE Improvement (Persistence - XGB)\nLead {lead}"
    )

    plt.savefig(
        f"xgb_results/improv_leadpng/improvement_lead_{lead}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    del model
    del X_test
    del rmse_map
    del pred_df
    del improvement_da

    gc.collect()

results_df = pd.DataFrame(results)

results_df.to_parquet("rmse_summary.parquet")

Lead 1: Validation=0.01016, Persistence=0.01848, XGB=0.01984
Lead 2: Validation=0.01436, Persistence=0.02329, XGB=0.02527
Lead 3: Validation=0.01911, Persistence=0.02836, XGB=0.03072
Lead 4: Validation=0.02414, Persistence=0.03372, XGB=0.03578
Lead 5: Validation=0.02859, Persistence=0.03848, XGB=0.03944
Lead 6: Validation=0.03316, Persistence=0.04307, XGB=0.04314
Lead 7: Validation=0.03711, Persistence=0.04710, XGB=0.04583


In [51]:
print(df_features.memory_usage(deep=True).sum() / 1024**3)

6.407370995730162


In [52]:
import psutil

print("Total RAM:", psutil.virtual_memory().total / 1024**3, "GB")
print("Available:", psutil.virtual_memory().available / 1024**3, "GB")

Total RAM: 62.40182113647461 GB
Available: 26.020671844482422 GB


In [53]:
importance_df = pd.DataFrame(index=features)
for lead in range(1, 8):

    model = XGBRegressor()
    model.load_model(f"xgb_results/model_info/xgb_lead_{lead}.json")

    fi = pd.Series(
        model.feature_importances_,
        index=features,
        name=f"lead_{lead}"
    )

    importance_df[f"lead_{lead}"] = fi

importance_df

,lead_1,lead_2,lead_3,lead_4,lead_5,lead_6,lead_7
sla_now,0.761720,0.733795,0.703003,0.661224,0.625121,0.591887,0.566763
sla_lag1,0.136043,0.140335,0.131412,0.133303,0.135763,0.141388,0.145604
sla_lag2,0.060347,0.063519,0.073492,0.082951,0.088650,0.091302,0.093812
sla_lag3,0.022071,0.030380,0.040907,0.047560,0.052129,0.056687,0.057183
sla_lag4,0.003869,0.007514,0.011572,0.013429,0.014293,0.014517,0.017291
sla_lag5,0.005990,0.006489,0.006660,0.008020,0.008290,0.009318,0.010272
sla_lag6,0.000847,0.001412,0.002135,0.003137,0.004072,0.004703,0.005036
sla_lag7,0.000414,0.001539,0.003383,0.005553,0.006594,0.007982,0.007674
sla_lag8,0.000607,0.001864,0.003615,0.006099,0.008159,0.009437,0.009572
sla_lag9,0.000472,0.001317,0.002425,0.003481,0.004768,0.006279,0.006352
